# Fine-tuning from MACE Pretrained Weights

This example shows how to initialize the LogP model from pretrained MACE weights.

Benefits:
- Faster convergence
- Better generalization
- Can freeze backbone and only train readout heads

In [1]:
import torch
import lightning as L
from pathlib import Path
from ase.build import bulk
from ase.io import write

from NPS.logp.models import LitLogPModel
from NPS.logp.data import PeriodicStructureDataModule

/opt/anaconda3/envs/nps-logp/lib/python3.11/site-packages/e3nn/o3/_wigner.py:10: UserWarning: Environment variable TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD detected, since the`weights_only` argument was not explicitly passed to `torch.load`, forcing weights_only=False.
  _Jd, _W3j_flat, _W3j_indices = torch.load(os.path.join(os.path.dirname(__file__), 'constants.pt'))


cuequivariance or cuequivariance_torch is not available. Cuequivariance acceleration will be disabled.


## 1. Load Pretrained MACE Weights

In [2]:
# Path to your pretrained MACE checkpoint
MACE_PATH = "./2024-01-07-mace-128-L2_epoch-199.pt"  

# Load checkpoint (weights_only=False needed for PyTorch 2.6+)
ckpt = torch.load(MACE_PATH, map_location='cpu', weights_only=False)

# Get state dict - depends on checkpoint format
if hasattr(ckpt, 'state_dict'):
    # It's a model object
    pretrained_state_dict = ckpt.state_dict()
elif isinstance(ckpt, dict) and 'state_dict' in ckpt:
    # Lightning checkpoint
    pretrained_state_dict = ckpt['state_dict']
elif isinstance(ckpt, dict) and 'model' in ckpt:
    # Wrapped in 'model' key
    pretrained_state_dict = ckpt['model']
elif isinstance(ckpt, dict):
    # Already a state dict
    pretrained_state_dict = ckpt
else:
    raise ValueError(f"Unknown checkpoint format: {type(ckpt)}")

print(f"Loaded {len(pretrained_state_dict)} parameters")

Loaded 90 parameters


## 2. Create Training Data

In [3]:
data_dir = Path("finetune_data")
data_dir.mkdir(exist_ok=True)

# BCC and FCC structures
write(data_dir / 'bcc.extxyz', [
    bulk('Fe', 'bcc', a=2.87, cubic=True) * (3,3,3),
    bulk('W', 'bcc', a=3.16, cubic=True) * (3,3,3),
])
write(data_dir / 'fcc.extxyz', [
    bulk('Cu', 'fcc', a=3.61, cubic=True) * (3,3,3),
    bulk('Al', 'fcc', a=4.05, cubic=True) * (3,3,3),
])

structure_types = ["bcc", "fcc"]

In [4]:
dm = PeriodicStructureDataModule(
    file_list=[str(data_dir / f"{s}.extxyz") for s in structure_types],
    cutoff=6.0,
    duplicate=20,
    batch_size=2,
    num_workers=0,
    structure_types=structure_types,
)

## 3. Create Model with Pretrained Weights

**Important:** Match `hidden_irreps` and `num_interactions` to the pretrained model!

From filename `mace-128-L2`:
- `128` → `hidden_irreps="128x0e+128x1o+128x2e"`
- `L2` → `num_interactions=2`

In [7]:
model = LitLogPModel(
    num_species=89,
    nclass=2,
    cutoff=6.0,
    hidden_irreps="128x0e+128x1o+128x2e",  # Match pretrained
    num_interactions=2,                      # Match pretrained (L2)
    sigma_max=0.15,
    sigma_logp_scale=0.15,
    learn_rate=1e-3,  
    pretrained_state_dict=pretrained_state_dict,
    freeze_backbone=True,  # Freeze encoder + message passing layers
)

n_params = sum(p.numel() for p in model.parameters())
n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total params: {n_params:,}")
print(f"Trainable: {n_trainable:,}")

/opt/anaconda3/envs/nps-logp/lib/python3.11/site-packages/torch/jit/_check.py:178: UserWarning: The TorchScript type system doesn't support instance-level annotations on empty non-base types in `__init__`. Instead, either 1) use a type annotation in the class body, or 2) wrap the type in `torch.jit.Attribute`.
  warnings.warn(
/opt/anaconda3/envs/nps-logp/lib/python3.11/site-packages/mace/modules/blocks.py:312: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  torch.tensor(atomic_energies, dtype=torch.get_default_dtype()),
/opt/anaconda3/envs/nps-logp/lib/python3.11/site-packages/torch/jit/_check.py:178: UserWarning: The TorchScript type system doesn't support instance-level annotations on empty non-base types in `__init__`. Instead, either 1) use a type annotation in the class body, or 2) wrap the type in `torch.jit.Attribute`.
  warnings.

Loading 89/90 pretrained parameters
Freezing backbone (encoder + interactions)
Trainable parameters: 4528/5727408 (0.1%)
Total params: 11,454,816
Trainable: 9,056


/opt/anaconda3/envs/nps-logp/lib/python3.11/site-packages/torch/jit/_check.py:178: UserWarning: The TorchScript type system doesn't support instance-level annotations on empty non-base types in `__init__`. Instead, either 1) use a type annotation in the class body, or 2) wrap the type in `torch.jit.Attribute`.
  warnings.warn(
/opt/anaconda3/envs/nps-logp/lib/python3.11/site-packages/torch/jit/_check.py:178: UserWarning: The TorchScript type system doesn't support instance-level annotations on empty non-base types in `__init__`. Instead, either 1) use a type annotation in the class body, or 2) wrap the type in `torch.jit.Attribute`.
  warnings.warn(
/opt/anaconda3/envs/nps-logp/lib/python3.11/site-packages/torch/jit/_check.py:178: UserWarning: The TorchScript type system doesn't support instance-level annotations on empty non-base types in `__init__`. Instead, either 1) use a type annotation in the class body, or 2) wrap the type in `torch.jit.Attribute`.
  warnings.warn(
/opt/anaconda

## 4. Train

In [8]:
trainer = L.Trainer(
    accelerator='cpu',
    max_epochs=5,
    enable_checkpointing=False,
    logger=False,
)

trainer.fit(model, dm)
print("Done!")

GPU available: True (mps), used: False
TPU available: False, using: 0 TPU cores
/opt/anaconda3/envs/nps-logp/lib/python3.11/site-packages/lightning/pytorch/trainer/setup.py:175: GPU available but not used. You can set it by doing `Trainer(accelerator='gpu')`.

  | Name      | Type             | Params | Mode  | FLOPs
---------------------------------------------------------------
0 | model     | LogPModelWrapper | 5.7 M  | train | 0    
1 | ema_model | AveragedModel    | 5.7 M  | train | 0    
---------------------------------------------------------------
9.1 K     Trainable params
11.4 M    Non-trainable params
11.5 M    Total params
45.819    Total estimated model params size (MB)
239       Modules in train mode
0         Modules in eval mode
0         Total Flops


Sanity Checking: |                                                                                            …

/opt/anaconda3/envs/nps-logp/lib/python3.11/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=7` in the `DataLoader` to improve performance.
/opt/anaconda3/envs/nps-logp/lib/python3.11/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)
/opt/anaconda3/envs/nps-logp/lib/python3.11/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=7` in the `DataLoader` to improve performance.


Training: |                                                                                                   …

Validation: |                                                                                                 …

...

Validation: |                                                                                                 …

Validation: |                                                                                                 …

Validation: |                                                                                                 …

`Trainer.fit` stopped: `max_epochs=5` reached.


Done!
